# This file is for Experimenting and testing logic

In [1]:
print("Hello")

Hello


In [12]:
import os
import json
import pandas as pd
import traceback
from dotenv import load_dotenv
from langchain_ollama import ChatOllama

In [16]:
load_dotenv(override=True)

LLM_MODEL = os.getenv("OLLAMA_MODEL")

print(f"LLM_MODEL: {LLM_MODEL}")

LLM_MODEL: gemma4


In [20]:
llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0.3,
    max_tokens=1000,
    streaming=True,
    verbose=True,
)

In [22]:
respone = llm.invoke("What is GenAI, and how it is different from traditional AI?")
print(f"Response: {respone.content}")

Response: This is one of the most important distinctions in modern technology, and understanding it is key to understanding the current AI revolution.

In short: **Traditional AI analyzes and classifies what already exists; Generative AI creates something entirely new.**

Here is a detailed breakdown of both concepts and how they differ.

---

## 🧠 Part 1: What is Generative AI (GenAI)?

**Generative AI** refers to a class of artificial intelligence models that are designed not just to process or analyze data, but to **generate novel, realistic content**.

Instead of being trained to recognize patterns and label existing information, GenAI learns the underlying *structure* and *distribution* of massive datasets (text, images, code, audio) so well that it can predict what the next most logical element should be.

### 💡 How It Works
GenAI models are typically trained on vast amounts of data to understand complex relationships. When you give them a prompt (an input), they don't look up an

In [80]:
from langchain_classic.llms import Ollama
from langchain_core.prompts.prompt import PromptTemplate
from langchain_classic.chains import LLMChain, SequentialChain
from langchain_community.callbacks.manager import get_openai_callback
import PyPDF2

In [81]:
RESPONSE_JSON_STRUCTURE = {
    '1': {
        'mcq': 'multiple choice question',
        'options': {
            'a': 'choice here',
            'b': 'choice here',
            'c': 'choice here',
            'd': 'choice here'
        },
        'correct_answer': 'correct choice here',
    },
    '2': {
        'mcq': 'multiple choice question',
        'options': {
            'a': 'choice here',
            'b': 'choice here',
            'c': 'choice here',
            'd': 'choice here'
        },
        'correct_answer': 'correct choice here',
    }
}

In [82]:
TEMPLATE_QUIZ_GENERATION_PROMPT = """
You are an expert in {subject} and have been asked to create a multiple-choice quiz based on the following text:
{text} 
create a quiz with {number} questions, each with 4 options. The quiz should be in {tone} tone.
Make sure the questions are not repeated and check all the questions to be conforming to the text provided. 
The output should be in JSON format as per the following structure: {response_json}
Ensure that the questions are clear, concise, and relevant to the text provided.
And ensure to make {number} questions in the quiz.
"""

In [83]:
quiz_generation_prompt_template = PromptTemplate(
    input_variables=['text', 'number', 'subject', 'tone', 'response_json'],
    template=TEMPLATE_QUIZ_GENERATION_PROMPT
)

In [84]:
quiz_chain = LLMChain(
    llm=llm,
    prompt=quiz_generation_prompt_template,
    output_key='quiz',
    verbose=True
)

In [85]:
TEMPLATE2 = """
You are an expert english grammarian and writer. Give me a multiple-choice quiz for {subject} students.
You need to evaluate the complexity of the questions and give a complete analysis of the quiz. Only use at max 50 words for complexity analysis. The quiz should be in {tone} tone.
If the quiz is not good enough, you need to give a complete analysis of the quiz and suggest improvements.
Update the quiz based on your analysis and suggestions and change the tone such that it is more suitable for the students.
Quiz_MCQs: 
{quiz}

Check from an expert English Writer of the above quiz:
"""

In [86]:
chain2_prompt_template = PromptTemplate(
    input_variables=['quiz', 'subject', 'tone'],
    template=TEMPLATE2
)

In [87]:
quiz_evaluation_chain = LLMChain(
    llm=llm,
    prompt=chain2_prompt_template,
    output_key='review',
    verbose=True
)

In [88]:
generate_evaluation_chain = SequentialChain(
    chains=[quiz_chain, quiz_evaluation_chain],
    input_variables=['text', 'number', 'subject', 'tone', 'response_json'],
    output_variables=['quiz', 'review'],
    verbose=True
)

In [89]:
file_path = "/Users/mdirfan/dev/python/GenAI-MCQ/data.txt"

In [90]:
with open(file_path, 'r') as f:
    TEXT = f.read()

In [91]:
print(TEXT)

The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied human cognitive systems contributed to the m

In [101]:
json.dumps(RESPONSE_JSON_STRUCTURE)

'{"1": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct_answer": "correct choice here"}, "2": {"mcq": "multiple choice question", "options": {"a": "choice here", "b": "choice here", "c": "choice here", "d": "choice here"}, "correct_answer": "correct choice here"}}'

In [92]:
with get_openai_callback() as cb:
    generated_quiz = generate_evaluation_chain(
        {
            "text": TEXT,
            "number": 10,
            "subject": "machine learning",
            "tone": "simple",
            "response_json": json.dumps(RESPONSE_JSON_STRUCTURE)
        }
    )
    print(f"Total Tokens Used: {cb.total_tokens}")
    print(f"Prompt Tokens Used: {cb.prompt_tokens}")
    print(f"Completion Tokens Used: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost:.6f}")



> Entering new SequentialChain chain...


> Entering new LLMChain chain...
Prompt after formatting:

You are an expert in machine learning and have been asked to create a multiple-choice quiz based on the following text:
The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]

The earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwor

In [102]:
generated_quiz

{'text': 'The term machine learning was coined in 1959 by Arthur Samuel, an IBM employee and pioneer in the field of computer gaming and artificial intelligence.[5][6] The synonym self-teaching computers was also used during this time period.[7][8]\n\nThe earliest machine learning program was introduced in the 1950s, when Samuel invented a computer program that calculated the chance of winning in checkers for each side, but the history of machine learning is rooted in decades of efforts to study human cognitive processes.[9] In 1949, Canadian psychologist Donald Hebb published the book The Organization of Behavior, in which he introduced a theoretical neural structure formed by certain interactions among nerve cells.[10] The Hebbian theory of neuron interaction set the groundwork for how many machine learning algorithms work, with connected artificial neurons changing the strength of their connections based on data.[9] Other researchers who have studied human cognitive systems contribu

In [103]:
quiz = generated_quiz.get('quiz')

In [94]:
review = generated_quiz.get('review')

In [95]:
print(review)

***As an expert English grammarian and writer, I have analyzed your quiz. While the grammar is technically correct, the tone is overly academic and dense, making it feel more like a graduate-level history exam than a student study guide.***

### Complexity Analysis
The questions test deep historical recall (names, dates, specific books), which increases cognitive load unnecessarily. The complexity is high due to factual density rather than conceptual difficulty. Focus should shift from memorizing facts to understanding fundamental ML concepts and their origins.

---

### Complete Analysis and Suggested Improvements

**Overall Assessment:** The quiz content is excellent for testing deep historical knowledge of Machine Learning, but the presentation and tone are unsuitable for a general student audience. It reads like an advanced academic paper's quiz section.

**Grammar/Style Critique:**
1. **Tone:** Too formal and dry. Students need encouragement and clear framing.
2. **Clarity:** Some

In [104]:
print(quiz)

```json
{
  "1": {
    "mcq": "Who coined the term 'machine learning' in 1959?",
    "options": {
      "a": "Donald Hebb",
      "b": "Alan Turing",
      "c": "Arthur Samuel",
      "d": "Ian Goodfellow"
    },
    "correct_answer": "c"
  },
  "2": {
    "mcq": "During the early period of machine learning, what synonym was also used for the field?",
    "options": {
      "a": "Artificial Intelligence",
      "b": "Self-teaching computers",
      "c": "Computational Cognition",
      "d": "Neural Pattern Recognition"
    },
    "correct_answer": "b"
  },
  "3": {
    "mcq": "Which researcher published the book 'The Organization of Behavior' in 1949, setting groundwork for neural theories?",
    "options": {
      "a": "Walter Pitts",
      "b": "Warren McCulloch",
      "c": "Nils Nilsson",
      "d": "Donald Hebb"
    },
    "correct_answer": "d"
  },
  "4": {
    "mcq": "What did Walter Pitts and Warren McCulloch propose that mirrored human thought processes?",
    "options": {
   

In [105]:
quiz_str = quiz.replace("```json", "").replace("```", "").strip()

In [110]:
quiz_str

'{\n  "1": {\n    "mcq": "Who coined the term \'machine learning\' in 1959?",\n    "options": {\n      "a": "Donald Hebb",\n      "b": "Alan Turing",\n      "c": "Arthur Samuel",\n      "d": "Ian Goodfellow"\n    },\n    "correct_answer": "c"\n  },\n  "2": {\n    "mcq": "During the early period of machine learning, what synonym was also used for the field?",\n    "options": {\n      "a": "Artificial Intelligence",\n      "b": "Self-teaching computers",\n      "c": "Computational Cognition",\n      "d": "Neural Pattern Recognition"\n    },\n    "correct_answer": "b"\n  },\n  "3": {\n    "mcq": "Which researcher published the book \'The Organization of Behavior\' in 1949, setting groundwork for neural theories?",\n    "options": {\n      "a": "Walter Pitts",\n      "b": "Warren McCulloch",\n      "c": "Nils Nilsson",\n      "d": "Donald Hebb"\n    },\n    "correct_answer": "d"\n  },\n  "4": {\n    "mcq": "What did Walter Pitts and Warren McCulloch propose that mirrored human thought proc

In [106]:
json_output = json.dumps(quiz_str, indent=4)

In [111]:
quiz = json.loads(quiz_str)

In [112]:
print(type(quiz))

<class 'dict'>


In [116]:
quiz_table_data = []
for key, value in quiz.items():
    mcq = value.get('mcq')
    options = " | ".join(
        [
            f"{opt_key}: {opt_value}" for opt_key, opt_value in value.get('options', {}).items()
        ]
    ),
    correct_answer = value.get('correct_answer')
    quiz_table_data.append([key, mcq, options, correct_answer])

In [117]:
quiz_table_data

[['1',
  "Who coined the term 'machine learning' in 1959?",
  ('a: Donald Hebb | b: Alan Turing | c: Arthur Samuel | d: Ian Goodfellow',),
  'c'],
 ['2',
  'During the early period of machine learning, what synonym was also used for the field?',
  ('a: Artificial Intelligence | b: Self-teaching computers | c: Computational Cognition | d: Neural Pattern Recognition',),
  'b'],
 ['3',
  "Which researcher published the book 'The Organization of Behavior' in 1949, setting groundwork for neural theories?",
  ('a: Walter Pitts | b: Warren McCulloch | c: Nils Nilsson | d: Donald Hebb',),
  'd'],
 ['4',
  'What did Walter Pitts and Warren McCulloch propose that mirrored human thought processes?',
  ('a: The checkers algorithm | b: A mathematical model of neural networks | c: Generative Adversarial Networks (GANs) | d: Reinforcement learning techniques',),
  'b'],
 ['5',
  "What experimental 'learning machine,' developed by Raytheon Company in the 1960s, was used to analyze signals like sonar a

In [119]:
quiz = pd.DataFrame(quiz_table_data)

In [120]:
quiz.to_csv("machine_learning_quiz.csv", index=False)